In [1]:
# Cella 1 — Installazione e import
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "google-cloud-storage", "scikit-learn", "-q"])


CompletedProcess(args=['/opt/anaconda3/envs/gcp/bin/python', '-m', 'pip', 'install', 'google-cloud-storage', 'scikit-learn', '-q'], returncode=0)

In [2]:
from google.cloud import storage
import pandas as pd

In [3]:
import os
PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
print(PROJECT_ID)

KeyError: 'GOOGLE_CLOUD_PROJECT'

In [4]:
import io
BUCKET     = f"ias-luigi-asprino-bucket"   # adattare al nome usato
client = storage.Client()
blob   = client.bucket(BUCKET).blob("iris.csv")
df     = pd.read_csv(io.BytesIO(blob.download_as_bytes()))
print(df.head())

   sepal.length  sepal.width  petal.length  petal.width variety
0           5.1          3.5           1.4          0.2  Setosa
1           4.9          3.0           1.4          0.2  Setosa
2           4.7          3.2           1.3          0.2  Setosa
3           4.6          3.1           1.5          0.2  Setosa
4           5.0          3.6           1.4          0.2  Setosa


In [5]:
X = df.drop("variety", axis=1)
y = df["variety"]

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, clf.predict(X_test)))

Accuracy: 1.0


In [8]:
MODEL_DIR  = os.environ.get("AIP_MODEL_DIR", "/tmp/model")  # standard Vertex AI
print(MODEL_DIR)

/tmp/model


In [ ]:
import joblib

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(clf, f"{MODEL_DIR}/model.joblib", protocol=5)
    
client.bucket(BUCKET).blob("models/model.joblib") \
      .upload_from_filename(f"{MODEL_DIR}/model.joblib")
print("Modello salvato su GCS.")

Modello salvato su GCS.
